# Flattening nested data

##### When you pull data from APIs or NoSQL databases, it rarely comes in a clean, flat table. Interviewers want to know you can handle real-world, messy JSON structures

# PySpark `explode()` Function

The `explode()` function is used to **transform array or map columns into multiple rows**. Each element in the array/map becomes a separate row, duplicating all other column values.

## Key Points:
* **Input**: A column containing arrays or maps
* **Output**: Multiple rows, one for each array element or map entry
* **Behavior with empty arrays**: `explode()` drops rows with empty arrays
* **Alternative**: Use `explode_outer()` to keep rows with empty/null arrays (returns null for the exploded column)

## Common Use Cases:
* Flattening nested data structures
* Normalizing JSON data with array fields
* Converting one-to-many relationships into separate rows
* Processing shopping cart items, tags, or any list-type data

In [0]:
from pyspark.sql.functions import explode, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

## Example 1: Exploding an Array Column

In this example, we have orders with multiple items stored as an array. We'll use `explode()` to create one row per item.

**Transformation**:
* **Before**: 3 rows (one per order with array of items)
* **After**: 6 rows (one per individual item)
* Order O-1001 with `["Laptop", "Mouse", "Keyboard"]` becomes 3 separate rows

In [0]:
# Sample Data: Order ID and an Array of Items
data = [
    ("O-1001", ["Laptop", "Mouse", "Keyboard"]),
    ("O-1002", ["Monitor"]),
    ("O-1003", ["Desk", "Chair"])
]

# Define Schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("items", ArrayType(StringType()), True)
])

df = spark.createDataFrame(data, schema)
df.show(truncate=False)

+--------+-------------------------+
|order_id|items                    |
+--------+-------------------------+
|O-1001  |[Laptop, Mouse, Keyboard]|
|O-1002  |[Monitor]                |
|O-1003  |[Desk, Chair]            |
+--------+-------------------------+



In [0]:
display(df)

order_id,items
O-1001,"List(Laptop, Mouse, Keyboard)"
O-1002,List(Monitor)
O-1003,"List(Desk, Chair)"


In [0]:
# The PySpark Solution
flattened_df = df.select("order_id", explode("items").alias("single_item"))

flattened_df.show()

+--------+-----------+
|order_id|single_item|
+--------+-----------+
|  O-1001|     Laptop|
|  O-1001|      Mouse|
|  O-1001|   Keyboard|
|  O-1002|    Monitor|
|  O-1003|       Desk|
|  O-1003|      Chair|
+--------+-----------+



In [0]:
display(flattened_df)

order_id,single_item
O-1001,Laptop
O-1001,Mouse
O-1001,Keyboard
O-1002,Monitor
O-1003,Desk
O-1003,Chair


## Example 2: Empty Arrays - `explode()` vs `explode_outer()`

### Critical Behavior Difference:
* **`explode()`**: **Drops rows** with empty or null arrays
* **`explode_outer()`**: **Keeps rows** with empty/null arrays and sets the exploded column to `null`

### In This Example:
* Order O-2001 has 2 items → 2 rows
* Order O-2002 has 1 item → 1 row  
* Order O-2003 has **empty array** `[]` (abandoned cart)
  * With `explode()`: O-2003 **disappears** from the result
  * With `explode_outer()`: O-2003 **remains** with item = null

**Best Practice**: Use `explode_outer()` when you need to preserve all original records for data integrity (e.g., tracking abandoned carts).

In [0]:
data = [
    ("O-2001", ["Tablet", "Case"]),
    ("O-2002", ["Headphones"]),
    ("O-2003", [])  # Empty order - customer abandoned cart
]
columns = ["order_id", "items"]

df = spark.createDataFrame(data, columns)

# Apply the explode function
df_exploded = df.select("order_id", explode("items").alias("item"))

df_exploded.show()

+--------+----------+
|order_id|      item|
+--------+----------+
|  O-2001|    Tablet|
|  O-2001|      Case|
|  O-2002|Headphones|
+--------+----------+



In [0]:
# This keeps O-2003 (empty order) in the dataset with a null item
df_safe = df.select("order_id", explode_outer("items").alias("item"))
df_safe.show()

+--------+----------+
|order_id|      item|
+--------+----------+
|  O-2001|    Tablet|
|  O-2001|      Case|
|  O-2002|Headphones|
|  O-2003|      NULL|
+--------+----------+



## Example 3: Exploding Map/Dictionary Columns

When you use `explode()` on a **map (dictionary)** column, it automatically creates **two columns**:
* `key`: The dictionary key
* `value`: The corresponding value

### In This Example:
* Order O-3001's map `{"Laptop": 1200, "Mouse": 25}` becomes:
  * Row 1: O-3001, Laptop, 1200
  * Row 2: O-3001, Mouse, 25
* Order O-3002's map `{"Monitor": 350}` becomes:
  * Row 1: O-3002, Monitor, 350

**Result**: Each key-value pair in the map becomes a separate row with `key` and `value` columns.

In [0]:
data_map = [
    ("O-3001", {"Laptop": 1200, "Mouse": 25}),
    ("O-3002", {"Monitor": 350})
]

df_map = spark.createDataFrame(data_map, ["order_id", "item_prices"])

# Explode generates 'key' and 'value' columns automatically
df_map_exploded = df_map.select("order_id", explode("item_prices"))

df_map_exploded.show()

+--------+-------+-----+
|order_id|    key|value|
+--------+-------+-----+
|  O-3001| Laptop| 1200|
|  O-3001|  Mouse|   25|
|  O-3002|Monitor|  350|
+--------+-------+-----+



## Summary: `explode()` Quick Reference

### Syntax
```python
from pyspark.sql.functions import explode, explode_outer

# For arrays
df.select("id", explode("array_column").alias("item"))

# For maps
df.select("id", explode("map_column"))  # Creates 'key' and 'value' columns

# To keep rows with empty/null arrays
df.select("id", explode_outer("array_column").alias("item"))
```

### Best Practices
1. **Always use `.alias()`** to give the exploded column a meaningful name
2. **Use `explode_outer()`** when data integrity requires preserving all rows
3. **Be aware of row multiplication**: exploding large arrays can significantly increase row count
4. **For maps**: the output columns are automatically named `key` and `value` (can rename with `alias()`)

### Related Functions
* `posexplode()`: Like explode, but also returns the position/index of each element
* `posexplode_outer()`: Outer version of posexplode
* `inline()`: Explodes array of structs into separate columns